In [ ]:
!pip install -q \
  transformers \
  datasets \
  peft \
  accelerate \
  bitsandbytes \
  sentencepiece


In [ ]:
#mounting drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#from datasets import load_dataset

#dataset = load_dataset("json", data_files="fine_tune.jsonl")
#dataset

In [ ]:
#loading base model Tamil LLaMA 7B

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_id = "abhinand/tamil-llama-7b-base-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
#importing instruction-response training data

from datasets import load_dataset
dataset = load_dataset("json", data_files="fine_tune.jsonl")["train"]


In [ ]:
#defining example format

def format_example(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Response:
{example['response']}"""
    return {"text": prompt}

dataset = dataset.map(format_example)


In [ ]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=dataset["train"].column_names)


In [ ]:
#Setting up the training parameters

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/tamil_poetry_lora",

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   

    num_train_epochs=2,            
    learning_rate=2e-4,

    fp16=True,
    logging_steps=50,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    report_to="none"
)


In [ ]:
# SFT initialization

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"]
)


In [ ]:
#training call

trainer.train()